# 使用 Graphiti 实现图记忆

> **把对话转化为时序知识图谱，让实体、关系和事实随时间演化。**

想象一块钉满索引卡的软木板，卡片之间用彩色细绳相连。每张卡片代表一个人、一个项目或一个概念。每根绳子代表一种关系：「管理」「依赖」「共事」。再想象每根绳子都挂着日期标签，标明它是什么时候钉上去的。如果一段关系变了，你不要拆掉旧绳子，而是加一根新绳子，把旧绳子标记为过时。这就是**时序知识图谱（temporal knowledge graph）**的工作方式。

大多数记忆系统把每条存储的事实当成独立的点。「Alice 是工程师」存在一个位置，「Alice 管理安全团队」存在另一个位置。如果你问「谁向 Alice 的经理汇报？」，扁平记忆没有办法沿着这条链追下去。**知识图谱（knowledge graph）**解决了这个问题。它把信息存成一个网络：**实体**（节点，比如人和项目）通过**关系**（边，比如「管理」或「参与」）连接起来。你可以沿着这些连接回答多步问题。

**Graphiti** 由 Zep 开发，增加了一个关键维度：**时序性**。每条边都带有时间戳，记录一段关系何时开始、最近一次何时得到确认、是否已被判定失效。这让智能体（agent）不仅能回答「现在什么为真？」，还能回答「上周什么为真？」和「这个事实变过吗？」

在本 notebook 中，你将从零构建一个基于 Graphiti 的记忆系统。你将：

1. 连接 Neo4j 图数据库并初始化 Graphiti。
2. 把对话 episode（带时间戳的对话轮次）摄取进知识图谱。
3. 用自然语言查询搜索图谱。
4. 探索时序边，观察事实如何演化。
5. 运行社区检测，找出知识聚类。
6. 把图谱接入智能体循环，回答关系型问题。

**读完本 notebook，你将理解：**
- Graphiti 的摄取流程如何从原始文本中抽取实体和关系。
- 为什么时序边对长期运行的智能体很重要。
- 图记忆何时优于扁平向量库，何时不是。

## 核心概念

- **知识图谱（knowledge graph）：** 一种数据结构，把信息存成**节点**（实体，比如人、项目、工具），节点之间用**边**（关系，比如「管理」或「依赖」）连接。可以把它看成一张结构化的事实地图。
- **时序知识图谱：** 每条边都带时间戳的知识图谱。你可以查询某个时间点上什么为真。过时的关系不会被删除，而是带着失效标记留在图谱里。
- **episode：** 输入给 Graphiti 的数据单元，通常是一轮对话或一个文档分块（chunk）。每个 episode 都有时间戳（叫 `reference_time`），把它锚定在时间轴上。
- **实体抽取（entity extraction）：** Graphiti 基于 LLM 的流程读取原始文本，识别出命名实体（人物、组织、工具、地点）的过程。
- **关系抽取（relationship extraction）：** 同一条流程还会识别实体之间的关系。比如从「Alice 加入了安全团队」中抽取 `Alice -> MEMBER_OF -> safety team` 这条关系。
- **实体消解（entity resolution）：** 当 Graphiti 发现一个实体看起来像已知的实体时，会把它们合并。「Alice」和「Dr. Alice Chen」可能消解为同一个节点。
- **图搜索：** 在知识图谱上做查询。Graphiti 支持混合检索，把语义相似度（基于含义的匹配）和 BM25 文本检索（基于关键词的匹配）结合起来。
- **多跳推理（multi-hop reasoning）：** 沿着关系链跨越多个节点。例如：User -> 任职于 -> Acme Corp -> 位于 -> San Francisco。
- **社区检测（community detection）：** 一种找出紧密相连节点聚类的算法。这些聚类通常代表有意义的群体（比如「工程团队」或「某项目用到的工具」）。
- **Neo4j：** 存储 Graphiti 数据的图数据库，使用一种叫 Cypher 的查询语言。你不需要直接写 Cypher，Graphiti 会替你处理。
- **失效（invalidation）：** 事实变化时，Graphiti 不会删除旧的边，而是给旧边打上 `invalid_at` 时间戳，再创建一条新边。历史因此得以保留。

## 架构

<p align="center">
  <img src="../../images/diagrams/24_graph_memory_graphiti.svg" alt="Graphiti 架构图" width="720"/>
</p>

**数据流：**

1. **对话 episode**（带时间戳的对话轮次）进入 Graphiti 的摄取流程。
2. **Episode 处理器**把文本交给**实体抽取器**和**关系抽取器**，两者都由 LLM 驱动。
3. **实体消解器**合并重复项、把共指引用链接起来（把「她」指向「Alice」），并让新实体与已有节点保持一致。
4. 新节点和时序边被写入 **Neo4j**。发生变化的关系会在旧边上得到失效时间戳，在新边上得到创建时间戳。
5. 查询时，智能体使用**查询层**：图搜索（语义 + 关键词混合）、社区检测（寻找聚类）和时序过滤（按时间划定结果范围）。
6. 检索到的实体、关系和事实以结构化文本的形式注入智能体的上下文窗口。

## 准备工作

你需要三样东西：

1. **一个运行中的 Neo4j 实例。** 最简单的方式是 Docker：`docker run -d -p 7687:7687 -p 7474:7474 -e NEO4J_AUTH=neo4j/password neo4j:latest`
2. **一个 OpenAI API key。** Graphiti 默认用 OpenAI 做实体/关系抽取和嵌入（embedding）。
3. **下面列出的 Python 包。**

在一个 `.env` 文件里设置这些环境变量：

```
LLM_API_KEY=sk-...
NEO4J_URI=bolt://localhost:7687
NEO4J_USER=neo4j
NEO4J_PASSWORD=password
```

In [ ]:
%pip install -q graphiti-core neo4j python-dotenv deepagents langchain-openai

导入 Graphiti SDK、标准库工具，并加载环境变量。

In [ ]:
import os
import asyncio
import json
from datetime import datetime, timezone, timedelta
from graphiti_core import Graphiti
from graphiti_core.nodes import EpisodeType, CommunityNode, EntityNode
from graphiti_core.edges import EntityEdge

from dotenv import load_dotenv

load_dotenv()  # 从 .env 读取 LLM_API_KEY / LLM_BASE_URL / LLM_MODEL_NAME

from langchain_openai import ChatOpenAI
from deepagents import create_deep_agent

assert os.getenv("LLM_API_KEY"), "Set LLM_API_KEY in your .env file"

# 任何 OpenAI 兼容端点都可以：用官方 OpenAI 时省略 base_url 即可
llm = ChatOpenAI(
    model=os.getenv("LLM_MODEL_NAME", "gpt-4o-mini"),
    api_key=os.getenv("LLM_API_KEY"),
    base_url=os.getenv("LLM_BASE_URL"),
)



def to_text(content) -> str:
    """把字符串或内容块列表统一转成纯文本。"""
    if isinstance(content, str):
        return content
    return "".join(
        block.get("text", "") for block in content if isinstance(block, dict)
    )


# Graphiti / Mem0 等框架默认读取 OPENAI_* 环境变量；
# 这里把统一端点映射过去，让框架内部调用也走同一个网关。
os.environ.setdefault("OPENAI_API_KEY", os.getenv("LLM_API_KEY", ""))
os.environ.setdefault("OPENAI_BASE_URL", os.getenv("LLM_BASE_URL", ""))
if os.getenv("LLM_MODEL_NAME"):
    os.environ.setdefault("OPENAI_MODEL", os.getenv("LLM_MODEL_NAME"))

print("✓ Model ready")


## 实现

我们分四步构建图记忆：

1. **初始化** Graphiti 并连接 Neo4j。
2. **摄取**事实不断演化的对话 episode。
3. 用自然语言**搜索**图谱。
4. **检测社区**，找出知识聚类。

### 第 1 步：初始化 Graphiti

把这一步想象成布置一块空白的软木板。你先搭好板子（Neo4j 数据库），贴上标签标明东西放在哪里（索引和约束），再准备好图钉（Graphiti 客户端）。

Graphiti 需要一个运行中的 Neo4j 实例。`build_indices_and_constraints()` 会创建 Graphiti 所需的数据库 schema：索引用于加速查询，约束用于保证数据完整性。

In [ ]:
async def initialize_graphiti() -> Graphiti:
    """连接 Neo4j 并准备图谱 schema。"""
    graphiti = Graphiti(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)
    await graphiti.build_indices_and_constraints()
    print(f"Connected to Neo4j at {NEO4J_URI}")
    print("Indices and constraints are ready.")
    return graphiti

graphiti = await initialize_graphiti()

### 第 2 步：摄取对话 episode

把每个 episode 想象成一篇日记。你写下发生了什么、什么时候发生的。时间久了，这本日记就讲出了事情如何变化的故事。Graphiti 读取每一篇日记，把新卡片（实体）和新绳子（关系）钉到软木板上。

我们会输入一段真实感很强的多会话对话，其中事实不断演化。Alice 先在一家公司任职，然后换团队、接手新项目。Graphiti 的流程会：

- 抽取实体（Alice、Anthropic、RLHF 团队等）
- 抽取关系（「Alice 在 Anthropic 工作」「Alice 在 RLHF 团队」）
- 消解重复项（不同 episode 里提到的「Alice」映射到同一个节点）
- 处理时序变化（Alice 换团队时，旧的边被标记失效）

In [ ]:
# 真实感的对话 episode，时间戳展示了事实如何演化
episodes = [
    {
        "body": "User: My name is Alice Chen. I work at Anthropic as a safety researcher.",
        "timestamp": datetime(2025, 1, 15, 10, 0, tzinfo=timezone.utc),
        "description": "User introduction",
    },
    {
        "body": "User: My manager is Bob Martinez. We're both on the RLHF team.",
        "timestamp": datetime(2025, 1, 20, 14, 30, tzinfo=timezone.utc),
        "description": "Team structure details",
    },
    {
        "body": "User: Our team uses PyTorch and we're training a reward model on the Helios cluster.",
        "timestamp": datetime(2025, 2, 5, 9, 0, tzinfo=timezone.utc),
        "description": "Technical stack details",
    },
    {
        "body": "User: I transferred from the RLHF team to the interpretability team last week. My new manager is Carol Park.",
        "timestamp": datetime(2025, 3, 1, 11, 0, tzinfo=timezone.utc),
        "description": "Team change announcement",
    },
    {
        "body": "User: On the interpretability team, we use JAX instead of PyTorch. I'm working on a new project called Lens.",
        "timestamp": datetime(2025, 3, 10, 16, 0, tzinfo=timezone.utc),
        "description": "New team technical details",
    },
    {
        "body": "User: Bob Martinez left Anthropic. He joined OpenAI as a research lead.",
        "timestamp": datetime(2025, 4, 2, 10, 0, tzinfo=timezone.utc),
        "description": "Personnel change",
    },
]

print(f"Prepared {len(episodes)} episodes spanning "
      f"{episodes[0]['timestamp'].strftime('%b %Y')} to "
      f"{episodes[-1]['timestamp'].strftime('%b %Y')}")

现在我们把每个 episode 输入 Graphiti。`add_episode` 方法是异步的。每次调用都会触发完整的抽取流程：实体抽取、关系抽取、实体消解和图谱更新。

**注意：** episode 必须按顺序逐个添加。每个 episode 都建立在前面的上下文之上。

In [ ]:
for i, episode in enumerate(episodes):
    result = await graphiti.add_episode(
        name=f"episode_{i:02d}",
        episode_body=episode["body"],
        source=EpisodeType.message,
        source_description=episode["description"],
        reference_time=episode["timestamp"],
    )
    ts = episode["timestamp"].strftime("%Y-%m-%d")
    print(f"Ingested episode {i} ({ts}): {episode['body'][:60]}...")

print(f"\nAll {len(episodes)} episodes ingested.")

### 第 3 步：搜索知识图谱

把图搜索想象成向一位读过所有书、还记得书与书之间联系的图书管理员提问。你问一个自然语言问题，管理员找到相关的索引卡，并沿着卡片之间的绳子追下去。

Graphiti 的 `search()` 方法执行混合检索，结合了：
- **语义相似度：** 找到含义与你的查询接近的边。
- **BM25 文本检索：** 找到与你的查询共享关键词的边。

返回结果是 `EntityEdge` 对象。每条边有一个 `fact` 字段（关系的自然语言陈述）和若干时序字段（`valid_at`、`invalid_at`）。

In [ ]:
async def search_and_display(graphiti_client: Graphiti, query: str, num_results: int = 5):
    """搜索图谱并打印带时序信息的结果。"""
    print(f"Query: {query}")
    print("-" * 60)

    results = await graphiti_client.search(query, num_results=num_results)

    if not results:
        print("  No results found.")
        return results

    for i, edge in enumerate(results):
        print(f"  [{i+1}] {edge.fact}")
        if edge.valid_at:
            print(f"      Valid from: {edge.valid_at.strftime('%Y-%m-%d')}")
        if edge.invalid_at:
            print(f"      Invalid at: {edge.invalid_at.strftime('%Y-%m-%d')}")
        else:
            print(f"      Status: current")
    print()
    return results

我们来试几个查询。注意图谱如何同时捕捉当前事实和历史事实。

In [ ]:
# 查询 1：当前团队归属
await search_and_display(graphiti, "What team is Alice on?")

# 查询 2：关系型问题（多跳）
await search_and_display(graphiti, "Who is Alice's manager?")

# 查询 3：工具/技术查询
await search_and_display(graphiti, "What tools does Alice's team use?")

### 第 4 步：探索时序边

事实变化时，Graphiti 的真正威力才显现出来。Alice 从 RLHF 团队调到了可解释性团队，Bob 离开了 Anthropic。我们来看看图谱如何表示这些变化。

被取代的时序边会带一个 `invalid_at` 时间戳。仍然有效的边，其 `invalid_at` 为 `None`。

In [ ]:
# 搜索 Bob，对比历史事实与当前事实
print("=== Bob Martinez: tracking a career change ===")
bob_results = await graphiti.search("Bob Martinez", num_results=10)

current_facts = []
historical_facts = []

for edge in bob_results:
    if edge.invalid_at is not None:
        historical_facts.append(edge)
    else:
        current_facts.append(edge)

print(f"\nCurrent facts ({len(current_facts)}):")
for edge in current_facts:
    print(f"  - {edge.fact}")

print(f"\nHistorical facts ({len(historical_facts)}):")
for edge in historical_facts:
    valid = edge.valid_at.strftime('%Y-%m-%d') if edge.valid_at else '?'
    invalid = edge.invalid_at.strftime('%Y-%m-%d') if edge.invalid_at else '?'
    print(f"  - {edge.fact}  (valid: {valid} to {invalid})")

### 第 5 步：社区检测

把社区想象成公司里的部门。看一眼公司的组织架构图，一起共事的人自然聚成一团。社区检测通过分析哪些节点连接密集，自动找出这些聚类。

Graphiti 用 Leiden 算法给节点分组，得到一组 `CommunityNode` 对象，每个对象都有名称和一个描述该聚类的摘要。

In [ ]:
# 从当前图谱状态构建社区
community_nodes, community_edges = await graphiti.build_communities()

print(f"Found {len(community_nodes)} communities:\n")
for community in community_nodes:
    print(f"Community: {community.name}")
    print(f"  Summary: {community.summary[:200]}")
    print()

## 示例运行：图驱动的智能体

现在把图谱接入智能体循环。这个智能体会：

1. 接收用户问题。
2. 在知识图谱中搜索相关事实。
3. 把这些事实注入系统提示词（prompt）。
4. 调用 LLM，生成有图谱依据的回答。

这是图增强生成的核心模式。图谱充当结构化的外部记忆，智能体在每次回答前都先查阅它。

In [ ]:
class GraphMemoryAgent:
    """一个把 Graphiti 知识图谱用作记忆的智能体。"""

    def __init__(self, graphiti_client: Graphiti, model):
        self.graphiti = graphiti_client
        self.model = model

    async def answer(self, question: str, num_facts: int = 10) -> str:
        """用从知识图谱检索到的事实回答问题。"""
        # 第 1 步：搜索图谱
        edges = await self.graphiti.search(question, num_results=num_facts)

        # 第 2 步：把检索到的事实格式化，放进提示词
        if edges:
            facts_text = "\n".join(
                f"- {edge.fact}"
                + (f" (valid from {edge.valid_at.strftime('%Y-%m-%d')})"
                   if edge.valid_at else "")
                + (f" [NO LONGER TRUE as of {edge.invalid_at.strftime('%Y-%m-%d')}]"
                   if edge.invalid_at else "")
                for edge in edges
            )
        else:
            facts_text = "No relevant facts found in the knowledge graph."

        system_prompt = (
            "You are a helpful assistant with access to a knowledge graph. "
            "Use the retrieved facts below to answer the user's question. "
            "If a fact is marked as no longer true, mention that it changed. "
            "If the facts don't cover the question, say so.\n\n"
            f"Retrieved facts:\n{facts_text}"
        )

        # 第 3 步：生成回答（图谱上下文每轮变化，所以每轮重建智能体）
        agent = create_deep_agent(model=self.model, system_prompt=system_prompt)
        result = agent.invoke({"messages": [{"role": "user", "content": question}]})
        return to_text(result["messages"][-1].content)

agent = GraphMemoryAgent(graphiti, model=llm)

我们用需要关系推理和时序感知的问题来测试这个智能体。这类问题正是扁平向量库难以应付的。

In [ ]:
test_questions = [
    "What team is Alice currently on, and who is her manager?",
    "Did Alice change teams? What was her previous team?",
    "Where does Bob Martinez work now? Has that changed recently?",
    "What tools and frameworks are used by Alice's current team?",
    "What project is Alice working on?",
]

for question in test_questions:
    print(f"Q: {question}")
    answer = await agent.answer(question)
    print(f"A: {answer}")
    print("-" * 60)

### 检查图谱结构

看看 Graphiti 实际构建了什么。我们会搜索一批边，打印源节点和目标节点的 UUID，观察连接结构。

In [ ]:
# 从图谱中取一批较宽泛的边
all_edges = await graphiti.search("Alice Anthropic team project", num_results=15)

print(f"Found {len(all_edges)} edges in the graph:\n")
print(f"{'Fact':<65} {'Status':<12} {'Valid From':<12}")
print("=" * 89)

for edge in all_edges:
    fact_display = edge.fact[:62] + "..." if len(edge.fact) > 65 else edge.fact
    status = "historical" if edge.invalid_at else "current"
    valid = edge.valid_at.strftime("%Y-%m-%d") if edge.valid_at else "unknown"
    print(f"{fact_display:<65} {status:<12} {valid:<12}")

### 向已有图谱添加新 episode

Graphiti 的一个关键特性是增量更新。新信息到来时，你不需要从头重建图谱，只需添加 episode，流程会把它们融入已有图谱。

In [ ]:
# 添加一个带最新信息的新 episode
new_episode = {
    "body": "User: Great news! The Lens project got approved for production deployment. Carol Park presented it to the leadership team.",
    "timestamp": datetime(2025, 5, 15, 9, 0, tzinfo=timezone.utc),
    "description": "Project status update",
}

await graphiti.add_episode(
    name="episode_06",
    episode_body=new_episode["body"],
    source=EpisodeType.message,
    source_description=new_episode["description"],
    reference_time=new_episode["timestamp"],
)
print("New episode ingested.")

# 搜索关于 Lens 的最新事实
print("\nUpdated facts about the Lens project:")
await search_and_display(graphiti, "Lens project status")

### 清理

关闭 Graphiti 连接，释放 Neo4j 资源。

In [ ]:
await graphiti.close()
print("Graphiti connection closed.")

## 权衡

### 图记忆占优的场景

- **关系型问题。**「谁管理 Alice？」「Alice 的团队用什么工具？」这类问题需要沿着连接遍历，扁平向量库无法追踪关系链。
- **时序推理。**「Alice 之前在哪个团队？」「Bob 换过工作吗？」这类问题需要关系上的时间戳。Graphiti 会记录事实何时成立、何时失效。
- **以实体为中心的召回。** 当智能体需要为单个实体拼出完整图景（Alice 的所有项目、团队、工具和同事）时，图遍历能把与该节点相连的一切收集齐。
- **长期运行的智能体。** 几周或几个月的对话里，事实会变化。图记忆通过边失效而不是覆盖旧数据来优雅地处理这些变化。

### 图记忆吃力的场景

- **基础设施开销。** 你需要一个运行中的 Neo4j 实例，也就是一个要部署、监控和备份的数据库。ChromaDB 这类向量库可以在进程内运行，不需要服务器。
- **摄取延迟。** 每次 `add_episode` 调用都会触发基于 LLM 的抽取，每个 episode 要花数秒而不是毫秒。对要求快速响应的实时聊天，你可能需要在后台异步摄取 episode。
- **抽取质量。** LLM 流程可能漏掉实体、幻觉出不存在的关系，或者无法合并重复项。图谱的质量取决于抽取的质量。
- **成本。** 每次摄取 episode 都要调用多次 LLM（实体抽取、关系抽取、消解），费用会累积。一个 100 个 episode 的对话，光是抽取用的 API 调用就可能花掉几美元。
- **简单回忆。** 如果你的智能体只需要「用户说过 X 的什么？」，带嵌入的向量库更快、更便宜、也更简单。这种直接查事实的场景，用图记忆是杀鸡用牛刀。

## 延伸阅读

- [Graphiti GitHub 仓库](https://github.com/getzep/graphiti)：开源代码、API 参考和示例 notebook，教你从对话构建时序知识图谱。
- [Zep 文档](https://docs.getzep.com?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：Graphiti 与更完整的 Zep 记忆平台的生产环境部署指南。
- [Neo4j 图数据库文档](https://neo4j.com/docs/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：驱动 Graphiti 的图数据库后端，覆盖 Cypher 查询、索引和图算法。
- [Ji et al., "A Survey on Knowledge Graphs," IEEE TNNLS, 2021](https://ieeexplore.ieee.org/document/9416312)：介绍知识图谱构建、嵌入方法和下游应用。
- [Packer et al., "MemGPT: Towards LLMs as Operating Systems," 2023 (arXiv:2310.08560)](https://arxiv.org/abs/2310.08560)：智能体自主管理记忆的相关工作，与图方法互补。
- [Kang et al., "Zep: A Temporal Knowledge Graph Architecture for Agent Memory," 2025 (arXiv:2501.13956)](https://arxiv.org/abs/2501.13956)：Graphiti 时序知识图谱设计背后的论文。

*← 上一篇：[23 - 工具化记忆](../23_memory_with_tools/) · 下一篇：[25 - Mem0 集成模式](../25_mem0_patterns/) →*

## 🧪 动手练习

三个小挑战，帮助你加深理解。每个大约需要 10-30 分钟。

### 挑战 1：探索社区
添加 10 个 episode 后，调用 `build_communities()` 并检查得到的社区节点。打印每个社区的名称和它包含的实体，讨论这些社区是否与你对话数据中的主题吻合。

### 挑战 2：图谱增长指标
每次调用 `add_episode()` 后，查询图谱中的实体节点总数、实体边总数和平均节点度。把这些指标沿着 episode 序列画出来，判断图谱是线性增长，还是随着交叉引用增多而加速增长。

### 挑战 3：时序矛盾检测
添加两个包含矛盾事实的 episode，针对同一个实体（例如先说「Alice works at Acme」再说「Alice works at Globex」）。用 `search()` 检索关于 Alice 的事实，检查旧边的 `expired_at` 时间戳是否被设置。这个实验演示了 18 时序记忆中探讨的时序失效思想。

![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--24-graph-memory-graphiti--graph-memory-graphiti)
